In [1]:
import numpy as np
import pandas as pd
import glob, os, sys
import matplotlib.pyplot as plt
import seaborn as sns
import scipy
import scipy.stats as st
import statsmodels.stats.api as sm

import Bio.PDB
from Bio import Seq, SeqIO
from Bio.PDB.MMCIFParser import MMCIFParser
from Bio.PDB.DSSP import make_dssp_dict
from Bio.PDB.Polypeptide import protein_letters_3to1

h37Rv = SeqIO.read("/n/data1/hms/dbmi/farhat/Sanjana/H37Rv/GCF_000195955.2_ASM19595v2_genomic.gbff", "genbank")
h37Rv_genes = pd.read_csv("/n/data1/hms/dbmi/farhat/Sanjana/H37Rv/mycobrowser_h37rv_genes_v4.csv")

os.chdir("../")
sys.path.append("utils")
from data_utils import *
from inSilicoMut_utils import *

who_variants = pd.read_csv("./data_processing/data_utils/WHO_catalog_V2.csv", header=[2]).reset_index(drop=True)
silent_lst = ['synonymous_variant', 'initiator_codon_variant', 'stop_retained_variant']

/tmp/ipykernel_30550/2979534123.py:20: DtypeWarning: Columns (36,37,99,100,102,103,106,108,112) have mixed types. Specify dtype option on import or set low_memory=False.
  who_variants = pd.read_csv("./data_processing/data_utils/WHO_catalog_V2.csv", header=[2]).reset_index(drop=True)


# Make distance maps

## Notes for AlphaFold structures:

Code to save only the high confidence coordinates

```
select high_confidence, b > 70
save ethA_alphaFold_highConf.pdb, high_confidence
```

<!-- 
<ul>
    <li></li>
</ul> -->

In [2]:
# # need to leave the Unnamed: 0 index column (don't save with index = False) because evcouplings.compare.distances.py reads in the dataframe with index_col = 0
# pncA_structure_coords = np.load("distance_maps/I6XD65.npy")
# pncA_distance_map = pd.read_csv("distance_maps/I6XD65.csv")

# katG_structure_coords = np.load("distance_maps/P9WIE5.npy")
# katG_distance_map = pd.read_csv("distance_maps/P9WIE5.csv")

# # check that this is a pairwise matrix, meaning that it's symmetric
# assert scipy.linalg.issymmetric(pncA_structure_coords)
# assert scipy.linalg.issymmetric(katG_structure_coords)

# # and also that the diagonals are all 0
# assert sum(np.diagonal(pd.DataFrame(pncA_structure_coords))) == 0
# assert sum(np.diagonal(pd.DataFrame(katG_structure_coords))) == 0

In [117]:
# Function to parse CIF file and extract necessary information
def extract_cif_info(cif_file):
    parser = MMCIFParser()
    structure = parser.get_structure('protein', cif_file)
    
    model = structure[0]
    
    # List to hold extracted information
    data = []
    
    # Extract information for each residue
    for chain in model:
        chain_id = chain.id
        for i, res in enumerate(chain):
            if res.id[0] == ' ':  # Exclude heteroatoms for now
                res_id = res.id[1]
                seqres_id = i + 1
                res_name = res.resname
                try:
                    one_letter_code = protein_letters_3to1[res_name]
                except KeyError:
                    one_letter_code = 'X'  # Unknown residue
                
                hetatm = res.id[0] != ' '
                coord = res['CA'].coord if 'CA' in res else None

                # # Get secondary structure assignment from DSSP -- not necessary for spatial clustering, and need to install additional dependencies, so skip for now
                # dssp_key = (chain_id, (' ', res_id, ' '))
                # if dssp_key in dssp_dict:
                #     sec_struct = dssp_dict[dssp_key][1]
                #     sec_struct_3state = 'H' if sec_struct in 'GHI' else 'E' if sec_struct == 'E' else 'C'
                # else:
                #     sec_struct = 'NA'
                #     sec_struct_3state = 'NA'
                sec_struct = 'NA'
                sec_struct_3state = 'NA'

                # chain index = 0 because there is only one chain
                # add 1 to len(data) to make it 1-indexed (in residue coordinate space, not index)
                data.append([
                    len(data) + 1, seqres_id, res_id, one_letter_code, res_name,
                    0, chain_id, sec_struct, sec_struct_3state, hetatm, coord
                ])
    
    # Create DataFrame
    columns = ['id', 'seqres_id', 'coord_id', 'one_letter_code',
               'three_letter_code', 'chain_index', 'chain_id', 'sec_struct',
               'sec_struct_3state', 'hetatm', 'coord']
    df = pd.DataFrame(data, columns=columns)
    return df

In [118]:
# mmcif_file = 'ethA_AlphaFold.cif'

# df_ethA_AF = extract_cif_info(mmcif_file)

# # residues 1 and 484-489 are low-confidence in alpha fold, so exclude
# df_ethA_AF_highConf = df_ethA_AF.query("id >= 2 & id <= 483").reset_index(drop=True)
# df_ethA_AF_highConf.to_csv("distance_maps/P9WNF9_AF.csv")

In [120]:
RNA_polymerase = extract_cif_info("spatial_clustering/PDB/5uhb.cif")

In [122]:
DNA_gyrase = extract_cif_info("spatial_clustering/PDB/5bs8.cif")

In [124]:
# A, C are gyrA; B, D are gyrB. E-H are DNA substrates
print(DNA_gyrase.chain_id.unique())

gyrA = DNA_gyrase.query("chain_id in ['A', 'C']")
gyrB = DNA_gyrase.query("chain_id in ['B', 'D']")

# chains A and C have the same residues represented
# chain D has two more residues than chain B, so take D
gyrA = gyrA.query("chain_id=='A'")
gyrB = gyrB.query("chain_id=='D'")

assert len(gyrA) == gyrA.coord_id.nunique()
assert len(gyrB) == gyrB.coord_id.nunique()

['A' 'B' 'C' 'D' 'E' 'F' 'G' 'H']


In [125]:
gyrB.query("coord_id in [674, 675]")

,id,seqres_id,coord_id,one_letter_code,three_letter_code,chain_index,chain_id,sec_struct,sec_struct_3state,hetatm,coord
1462,1463,246,674,D,ASP,0,D,NA,NA,False,"[37.362, -23.997, 45.799]"
1463,1464,247,675,V,VAL,0,D,NA,NA,False,"[34.732, -26.002, 47.704]"


In [96]:
# some sequence inconsistencies, but it's only 3 residues at the end, so maybe some random mutation or poor sequencing happened. Just exclude them
gyrA_test_seq = dict(zip(gyrA['coord_id'], gyrA['one_letter_code']))
gyrB_test_seq = dict(zip(gyrB['coord_id'], gyrB['one_letter_code']))

In [106]:
print(min(list(gyrA_test_seq.keys())), max(list(gyrA_test_seq.keys())))

for pos, residue in gyrA_test_seq.items():
    if residue != gyrA_protein_seq[pos-1]:
        print(pos, residue, gyrA_protein_seq[pos-1])

15 501
501 I A


In [107]:
print(min(list(gyrB_test_seq.keys())), max(list(gyrB_test_seq.keys())))

for pos, residue in gyrB_test_seq.items():
    if residue != gyrB_protein_seq[pos-1]:
        print(pos, residue, gyrB_protein_seq[pos-1])

424 675
424 N R
425 A E


In [118]:
gyrA = gyrA.query("coord_id != 501")
gyrA['id'] = gyrA['coord_id']

# rename these natural numbers
gyrA['seqres_id'] = np.arange(1, len(gyrA)+1)

gyrA.to_csv("distance_maps/5BS8_gyrA.csv")
get_distance_map_coordinates(gyrA, "5BS8_gyrA")

(485, 485)


In [119]:
gyrB = gyrB.query("coord_id not in [424, 425]")
gyrB['id'] = gyrB['coord_id']

# rename these natural numbers
gyrB['seqres_id'] = np.arange(1, len(gyrB)+1)
gyrB.to_csv("distance_maps/5BS8_gyrB.csv")
get_distance_map_coordinates(gyrB, "5BS8_gyrB")

(245, 245)


In [117]:
gyrA

,id,seqres_id,coord_id,one_letter_code,three_letter_code,chain_index,chain_id,sec_struct,sec_struct_3state,hetatm,coord
0,15,1,15,I,ILE,0,A,NA,NA,False,"[17.28, 33.757, -7.377]"
1,16,2,16,E,GLU,0,A,NA,NA,False,"[17.376, 36.749, -5.032]"
2,17,3,17,P,PRO,0,A,NA,NA,False,"[20.434, 39.04, -4.893]"
3,18,4,18,V,VAL,0,A,NA,NA,False,"[21.978, 40.14, -1.578]"
4,19,5,19,D,ASP,0,A,NA,NA,False,"[24.789, 42.634, -0.962]"
...,...,...,...,...,...,...,...,...,...,...,...
480,496,481,496,T,THR,0,A,NA,NA,False,"[42.004, 33.63, 50.569]"
481,497,482,497,R,ARG,0,A,NA,NA,False,"[44.022, 35.677, 53.058]"
482,498,483,498,I,ILE,0,A,NA,NA,False,"[44.317, 39.439, 52.546]"
483,499,484,499,I,ILE,0,A,NA,NA,False,"[47.655, 41.077, 53.362]"


In [24]:
# from PDB, chain C = rpoB, but the number is slightly off too
# residue 7 in PDB is residue 1 in H37Rv, so subtract 6
rpoB = RNA_polymerase.query("chain_id=='C'").reset_index(drop=True)
rpoB['coord_id'] -= 6
rpoB['id'] = rpoB['coord_id']

In [25]:
# did some manual checks
rpoB_protein_seq = h37Rv.seq[759807-1:763325].translate()


In [29]:
rpoB_protein_seq[21:1147]

Seq('SNNSVPGAPNRVSFAKLREPLEVPGLLDVQTDSFEWLIGSPRWRESAAERGDVN...EDE')

In [82]:
# from PDB, chain C = rpoB, but the number is slightly off too
# residue 7 in PDB is residue 1 in H37Rv, so subtract 6
rpoB = RNA_polymerase.query("chain_id=='C'").reset_index(drop=True)
rpoB['coord_id'] -= 6
rpoB['id'] = rpoB['coord_id']

rpoB.to_csv("distance_maps/5UHB.csv")

# did some manual checks
rpoB_protein_seq = h37Rv.seq[759807-1:763325].translate()
gyrA_protein_seq = h37Rv.seq[7301:9818].translate()
gyrB_protein_seq = h37Rv.seq[5239:7267].translate()

assert rpoB_protein_seq[-1] == '*'
assert gyrA_protein_seq[-1] == '*'
assert gyrB_protein_seq[-1] == '*'

In [33]:
get_distance_map_coordinates(rpoB, "5UHB")

(1126, 1126)


In [31]:
def calculate_pairwise_distances(coords):
    coords = np.array([coord for coord in coords if coord is not None])
    distances = np.linalg.norm(coords[:, np.newaxis] - coords, axis=-1)
    return distances

In [32]:
def get_distance_map_coordinates(df, dm_name):
    
    df.to_csv(f"distance_maps/{dm_name}.csv")

    assert sum(pd.isnull(df['coord'])) == 0
    
    # the coordinates are for the alpha carbon atom in each residue
    ca_coords = list(df['coord'])
    ca_distances = calculate_pairwise_distances(ca_coords)
    
    np.save(f"distance_maps/{dm_name}.npy", ca_distances)
    
    # pairwise matrix check
    assert scipy.linalg.issymmetric(ca_distances)
    assert sum(np.diagonal(pd.DataFrame(ca_distances))) == 0
    
    print(ca_distances.shape)

# Results of Clustering on Averaged Fold Change MIC Predictions from Site-Saturation Mutagenesis

In [174]:
def get_significant_GeO_scores(gene, pval_thresh=0.05):

    residue_data = pd.read_csv(f"spatial_clustering/{gene}/values_to_cluster.csv")

    GeO_scores = pd.read_csv(f"spatial_clustering/{gene}/G_scores.csv").merge(residue_data, on='residue')
    
    # these are GeO scores for each residue after shuffling the residues
    # perform a permutation test to see if the GeO score for each residue is significantly different from the the null
    GeO_permutation_results = pd.read_csv(f"spatial_clustering/{gene}/random_GeO_iterations_10000.csv.gz", compression="gzip", index_col=[0])
    
    for i, row in GeO_scores.iterrows():
    
        residue = row['residue']
        GeO_score = row['G_score']
    
        # what proportion of the permuted Getis-Ord statistics are at least as extreme as the Getis-Ord statistic for a given residue
        # positive GeO score is hot spot (R- or S-associated, depending on the prefix), negative GeO score is cold spot (neutral mutations)
        if GeO_score > 0:
            pvalue = np.mean(GeO_permutation_results.loc[residue, :].values >= GeO_score)
        else:
            pvalue = np.mean(GeO_permutation_results.loc[residue, :].values <= GeO_score)
    
        GeO_scores.loc[i, "pval"] = pvalue

    _, bh_pvals, _, _ = sm.multipletests(GeO_scores["pval"], method='fdr_bh', is_sorted=False, returnsorted=False)
    _, bonferroni_pvals, _, _ = sm.multipletests(GeO_scores["pval"], method='bonferroni', is_sorted=False, returnsorted=False)
    
    GeO_scores['BH_pval'] = bh_pvals
    GeO_scores['Bonferroni_pval'] = bonferroni_pvals
    
    GeO_scores.loc[(GeO_scores['BH_pval'] <= pval_thresh) & (GeO_scores['G_score'] > 0) & (GeO_scores['average'] > 0), 'clustering_result'] = 1

    # significant negative G score indicates cold spot (clustering of low values)
    GeO_scores.loc[(GeO_scores['BH_pval'] <= pval_thresh) & (GeO_scores['G_score'] < 0) & (GeO_scores['average'] < 0), 'clustering_result'] = -1

    # significant negative G score indicates cold spot (clustering of low values), but if the average prediction for that residue is positive, then it's clustering of neutral mutations
    # GeO_scores.loc[(GeO_scores['BH_pval'] <= pval_thresh) & (GeO_scores['G_score'] < 0) & (GeO_scores['average'] > 0), 'clustering_result'] = 0
    
    print(GeO_scores['clustering_result'].value_counts())

    # save and return it for future use
    GeO_scores.to_csv(f"./supplement/{gene}_GeO_scores.csv", index=False)
    
    return GeO_scores

In [175]:
catalytic_triad = [8, 96, 138]
iron_coordinating = [49, 51, 57, 71]

gene = 'pncA'
df_pncA = get_significant_GeO_scores(gene)

clustering_result
 1.0    25
-1.0     5
Name: count, dtype: int64


In [176]:
df_pncA.query("residue in @catalytic_triad")

,residue,G_score,average,pval,BH_pval,Bonferroni_pval,clustering_result
6,8,40.094698,3.809655,0.0045,0.027600,0.8280,1.0
94,96,46.986801,0.244966,0.0014,0.012267,0.2576,1.0
136,138,51.151384,0.672616,0.0001,0.006133,0.0184,1.0


In [177]:
df_pncA.query("residue in @iron_coordinating")

,residue,G_score,average,pval,BH_pval,Bonferroni_pval,clustering_result
47,49,36.486874,2.136783,0.0090,0.042462,1.0000,1.0
49,51,29.451894,3.841505,0.0266,0.078942,1.0000,NaN
55,57,55.691506,3.420377,0.0002,0.006133,0.0368,1.0
69,71,33.316769,2.087974,0.0134,0.057340,1.0000,NaN


In [178]:
# Arg104, Trp107, and His108 in a pocket distal to the heme, and His270, Trp321, and Asp381 in a pocket proximal to the heme. A covalently linked “MYW catalytic triad” is formed by the conserved residues, Met255, Tyr229, and Trp107

gene = 'katG'
df_katG = get_significant_GeO_scores(gene)

clustering_result
 1.0    104
-1.0     70
Name: count, dtype: int64


In [179]:
gene = 'ethA'
df_ethA = get_significant_GeO_scores(gene)

clustering_result
1.0    31
Name: count, dtype: int64


From this link: https://www.uniprot.org/uniprotkb/P9WNF9/entry

<ul>
    <li>FAD binding sites: 15, 36, 44-47, 56, 104</li>
    <li>NADP+ binding sites: 54-56, 183-189, 207-208</li>
    <li>Transition state stabilizer: 292</li>
</ul>

In [180]:
if 'annotation' in df_ethA.columns:
    del df_ethA['annotation']

# predicted binding site
FAD_binding_sites = [15, 36, 44, 45, 46, 47, 56, 104]
NADP_binding_site = [54, 55, 56, 183, 184, 185, 186, 187, 188, 189, 207, 208]
# transition_state_stabilizer = [292]

df_ethA.loc[df_ethA['residue'].isin(FAD_binding_sites), 'annotation'] = 'FAD binding'
df_ethA.loc[df_ethA['residue'].isin(NADP_binding_site), 'annotation'] = 'NADP+ binding'
# df_ethA.loc[df_ethA['residue'].isin(transition_state_stabilizer), 'annotation'] = 'transition state stabilizer'
df_ethA['annotation'] = df_ethA['annotation'].replace('nan', np.nan)

len(df_ethA.query("(residue in @FAD_binding_sites | residue in @NADP_binding_site) & BH_pval <= 0.05").sort_values("G_score", ascending=False)[['residue', 'G_score', 'annotation', 'BH_pval']]), len(df_ethA.dropna(subset='annotation'))

(12, 19)

In [181]:
# moxifloxacin
df_gyrA = get_significant_GeO_scores('gyrA')
df_gyrB = get_significant_GeO_scores('gyrB')

clustering_result
-1.0    74
 1.0    32
Name: count, dtype: int64
clustering_result
 1.0    8
-1.0    2
Name: count, dtype: int64


In [182]:
df_gyrA.dropna(subset='clustering_result')

,residue,G_score,average,pval,BH_pval,Bonferroni_pval,clustering_result
36,51,84.974318,0.003571,0.0052,0.016813,1.0000,1.0
40,55,69.199391,0.009741,0.0120,0.033448,1.0000,1.0
58,73,83.205315,0.066902,0.0054,0.017230,1.0000,1.0
59,74,103.735566,0.410705,0.0013,0.005254,0.6305,1.0
60,75,86.578329,0.054736,0.0054,0.017230,1.0000,1.0
...,...,...,...,...,...,...,...
447,463,-45.554776,-0.059118,0.0000,0.000000,0.0000,-1.0
453,469,-48.984870,-0.024173,0.0000,0.000000,0.0000,-1.0
454,470,-47.327707,-0.058965,0.0000,0.000000,0.0000,-1.0
456,472,-47.984115,-0.114745,0.0000,0.000000,0.0000,-1.0


In [183]:
df_gyrB.dropna(subset='clustering_result')

,residue,G_score,average,pval,BH_pval,Bonferroni_pval,clustering_result
67,498,99.022636,0.092786,0.0000,0.000000,0.0000,1.0
68,499,103.089351,1.952799,0.0000,0.000000,0.0000,1.0
69,500,143.682484,1.961800,0.0000,0.000000,0.0000,1.0
70,501,117.514616,2.858576,0.0000,0.000000,0.0000,1.0
71,502,118.432033,0.489068,0.0000,0.000000,0.0000,1.0
72,503,115.755279,0.175148,0.0000,0.000000,0.0000,1.0
73,504,80.208306,1.355059,0.0001,0.003500,0.0245,1.0
74,505,73.770761,0.089808,0.0007,0.019056,0.1715,1.0
243,674,-21.369062,-0.004242,0.0012,0.029400,0.2940,-1.0
244,675,-21.112518,-0.002218,0.0006,0.018375,0.1470,-1.0


In [184]:
df_rpoB = get_significant_GeO_scores('rpoB', pval_thresh=0.05)

clustering_result
1.0    88
Name: count, dtype: int64


In [185]:
# how many of the 27 RRDR sites are hot spots
print(len(df_rpoB.query("clustering_result==1")))

# wow, all of them
print(len(df_rpoB.query("clustering_result==1 & residue >= 426 & residue <= 452")))

88
27


In [186]:
df_rpoB.query("clustering_result==1").residue.min(), df_rpoB.query("clustering_result==1").residue.max()

(155, 674)

In [187]:
def print_pymol_selection_commands(df, coord_adj=0, chain_name='A', R_color='firebrick', S_color='marine', neutral_color='lightorange'):

    pval_col = 'BH_pval'
    
    # reset the coloring to gray
    print("select all")
    print("color gray80, all\n")

    R_hot_spots = [str(num + coord_adj) for num in df.query("clustering_result==1").residue.values]
    R_hot_spots = '+'.join(R_hot_spots)

    S_hot_spots = [str(num + coord_adj) for num in df.query("clustering_result==-1").residue.values]
    S_hot_spots = '+'.join(S_hot_spots)

    if len(R_hot_spots) > 0:
        print(f"select R_hot_spots, resi {R_hot_spots} and chain {chain_name}")
        print(f"color {R_color}, R_hot_spots\n")

    if len(S_hot_spots) > 0:
        print(f"select S_hot_spots, resi {S_hot_spots} and chain {chain_name}")
        print(f"color {S_color}, S_hot_spots\n")

    # cold_spots = [str(num + coord_adj) for num in df.query("clustering_result==0").residue.values]
    # cold_spots = '+'.join(cold_spots)

    # if len(cold_spots) > 0:
    #     print(f"select cold_spots, resi {cold_spots} and chain {chain_name}")
    #     print(f"color {neutral_color}, cold_spots\n")

In [165]:
print_pymol_selection_commands(df_pncA)

select all
color gray80, all

select R_hot_spots, resi 7+8+49+50+55+56+57+58+59+67+68+96+102+132+134+135+136+137+138+139+140+141+142+143+181 and chain A
color firebrick, R_hot_spots

select S_hot_spots, resi 25+26+28+29+30 and chain A
color marine, S_hot_spots

select cold_spots, resi 32+33+34+36+37+38+39+40+42 and chain A
color lightorange, cold_spots



In [13]:
print("select catalytic_triad, resi 8+96+138 and chain A")
print("color magenta, catalytic_triad\n")

print("select iron_coordinating, resi 49+51+57+71 and chain A")
print("color cyan, iron_coordinating")

select catalytic_triad, resi 8+96+138 and chain A
color magenta, catalytic_triad

select iron_coordinating, resi 49+51+57+71 and chain A
color cyan, iron_coordinating


In [114]:
print_pymol_selection_commands(df_ethA)

select all
color gray80, all

select R_hot_spots, resi 44+45+46+47+48+49+50+52+53+54+55+77+78+163+164+165+166+167+184+185+186+187+188+189+190+300+301+302+341+342+344 and chain A
color firebrick, R_hot_spots



In [166]:
print_pymol_selection_commands(df_katG, chain_name='A')

select all
color gray80, all

select R_hot_spots, resi 86+87+88+89+90+91+92+93+94+95+96+97+98+99+100+101+102+103+104+105+106+107+108+109+110+111+112+113+120+121+122+123+124+125+126+127+128+129+132+133+134+135+136+137+138+139+140+141+142+143+144+145+146+147+148+149+150+161+162+165+166+190+228+231+232+233+263+265+270+273+274+276+277+278+284+286+287+288+297+298+299+300+301+302+307+308+309+310+311+312+313+314+315+316+317+318+326+367+368+378+416+418+419+420 and chain A
color firebrick, R_hot_spots

select S_hot_spots, resi 445+446+447+448+449+450+455+456+458+466+467+468+469+470+504+506+507+509+510+511+512+513+514+515+521+522+523+525+527+528+529+530+531+532+533+542+547+549+554+555+556+557+558+559+562+563+564+565+567+641+651+652+662+664+665+666+677+678+680+681+682+683+685+712+717+718+719+720+721+722 and chain A
color marine, S_hot_spots

select cold_spots, resi 451+452+453+454+457+459+460+461+462+463+464+465+471+472+473+474+503+516+517+518+519+520+524+526+534+535+536+537+538+539+540+543+546+5

In [169]:
print_pymol_selection_commands(df_gyrA, chain_name='A')

select all
color gray80, all

select R_hot_spots, resi 51+55+73+74+75+76+77+78+79+80+81+82+83+84+85+86+87+88+89+90+91+92+93+94+95+96+97+98+124+125+128+130 and chain A
color firebrick, R_hot_spots

select S_hot_spots, resi 174+192+193+194+195+199+200+201+202+203+204+205+206+207+208+211+212+213+214+215+216+219+237+238+240+241+244+357+364+365+366+367+368+369+370+371+372+373+374+375+376+377+378+379+380+392+393+395+396+398+401+402+404+420+421+423+425+426+431+452+454+455+456+457+458+459+460+461+462+463+469+470+472+473 and chain A
color marine, S_hot_spots

select cold_spots, resi 144+147+148+149+169+170+171+196+197+198+209+210+217+218+220+221+222+223+224+225+226+242+243+245+246+356+358+359+360+361+362+363+381+382+383+384+385+386+387+388+389+390+391+394+397+399+400+403+405+418+422+424+427+428+429+430+432+433+434+435+448+449+450+451+453+464+465+466+467+468+471+474+475+476+477+478+479+480+481+482+483+484+485+486+487+488+489+491+492+493+494+498+500 and chain A
color lightorange, cold_spots



In [170]:
# select gyrB_R_hot_spots, (resi 498+499+500+501+502+503+504+505 and chain B) or (resi 498+499+500+501+502+503+504+505 and chain D)
# select gyrB_S_hot_spots, (resi 674+675 and chain B) or (resi 674+675 and chain D)
print_pymol_selection_commands(df_gyrB, chain_name='B')

select all
color gray80, all

select R_hot_spots, resi 498+499+500+501+502+503+504+505 and chain B
color firebrick, R_hot_spots

select S_hot_spots, resi 674+675 and chain B
color marine, S_hot_spots



In [149]:
print_pymol_selection_commands(df_rpoB, coord_adj=6, chain_name='C')

select all
color gray80, all

select R_hot_spots, resi 161+166+167+168+171+172+173+174+175+176+177+178+179+180+377+378+379+380+381+382+383+384+385+430+431+432+433+434+435+436+437+438+439+440+441+442+443+444+445+446+447+448+449+450+451+452+453+454+455+456+457+458+459+460+461+462+465+468+472+483+484+485+486+487+488+489+490+491+492+493+494+495+496+497+498+499+500+501+502+588+589+590+609+612+613+640+678+680 and chain C
color firebrick, R_hot_spots

